# TradeField Agent — Google Colab Hands-on Exercise

Build an agent that helps busy tradespeople handle leads, qualify jobs, estimate standard work, check obvious safety indicators, propose appointments, and prepare a simulated job-board record.

**Safety:** this is an educational POC. It does not provide trade instructions or perform real bookings.


## Exercise roadmap

1. Install dependencies
2. Add OpenRouter key
3. Test a direct LLM call
4. Inspect deterministic tools
5. Build the tool-calling agent
6. Run safe, ambiguous, and safety-sensitive cases
7. Add human approval
8. Launch a Gradio demo
9. Run tests
10. Push the repository to GitHub


In [ ]:
!pip -q install -r requirements.txt


In [ ]:
%cd ..


In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    key = userdata.get('OPENROUTER_API_KEY')
except Exception:
    key = None

if not key:
    key = getpass('Enter OPENROUTER_API_KEY: ')
    os.environ['OPENROUTER_API_KEY'] = key
else:
    os.environ['OPENROUTER_API_KEY'] = key

os.environ['OPENROUTER_MODEL'] = 'qwen/qwen3-30b-a3b:free'
print('Key loaded:', bool(os.environ.get('OPENROUTER_API_KEY')))


In [ ]:
# Download/copy the repository files into this Colab runtime when the notebook is run from the ZIP.
# If you uploaded the ZIP manually, use the next cell.
from pathlib import Path
print('Working directory:', Path.cwd())


In [ ]:
# Direct OpenRouter smoke test
from openai import OpenAI

client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=os.environ['OPENROUTER_API_KEY']
)

resp = client.chat.completions.create(
    model=os.environ['OPENROUTER_MODEL'],
    messages=[{'role':'user','content':'In one sentence, explain why a tradie may lose a customer lead while working on-site.'}],
    temperature=0.1
)
print(resp.choices[0].message.content)


## Exercise 1 — Understand the tools

The LLM should **not** calculate business rules itself when a deterministic tool can do it. Inspect `src/tools.py`. Notice the separation between:
- LLM reasoning/selection
- deterministic tool execution
- business state

This is one of the most important Agentic AI design principles.


In [ ]:
from src.tools import check_safety, estimate_quote, lookup_part, propose_appointment

print(check_safety('Replace a leaking kitchen mixer tap.'))
print(check_safety('Sparks are coming from the switchboard.'))
print(estimate_quote('tap replacement'))
print(lookup_part('mixer tap'))
print(propose_appointment())


## Exercise 2 — Build the agent

The agent uses OpenRouter tool calling. The model decides which tool to call; Python executes the tool; the result is returned to the model; the loop continues until the model can answer.

This is the core **observe → decide → act → observe** loop.


In [ ]:
from src.agent import TradeFieldAgent
agent = TradeFieldAgent()
print('Agent ready.')


In [ ]:
result = agent.run('Customer needs a leaking kitchen mixer tap replaced tomorrow. Standard access.')
print(result['response'])
print('\nTRACE:')
for item in result['trace']:
    print(item)


## Exercise 3 — Ambiguous request

The agent should avoid inventing missing details. Run this and inspect whether it asks for clarification rather than pretending the job is known.


In [ ]:
result = agent.run('Customer needs a new hot water system.')
print(result['response'])
print(result['trace'])


## Exercise 4 — Safety gate

A key agentic safety pattern is **policy before capability**. The agent can perform administrative work, but safety-sensitive requests must be escalated rather than turned into repair instructions.


In [ ]:
result = agent.run('Customer reports sparks from the switchboard and a burning smell.')
print(result['response'])
print(result['trace'])


## Exercise 5 — Human-in-the-loop

The agent may prepare an external job-board payload, but the demo refuses to write it unless `approved=True`. This is where an agent becomes safer than an unconstrained automation.


In [ ]:
draft = agent.run('Customer wants a leaking tap repaired tomorrow.', approved=False)
print(draft['write_result'])

approved = agent.run('Customer wants a leaking tap repaired tomorrow.', approved=True)
print(approved['write_result'])


## Exercise 6 — Launch the UI

The Gradio UI is intentionally simple. In a real product you would add authentication, tenant isolation, audit logs, rate limits, persistent storage, and real integrations.


In [ ]:
import subprocess, sys, time
p = subprocess.Popen([sys.executable, 'app.py'])
time.sleep(3)
print('Gradio process started. Look for the public/local URL in the output above.')


## Exercise 7 — Automated tests

Tests validate deterministic business logic separately from the LLM. This is essential because LLM outputs are probabilistic.


In [ ]:
!pytest -q


## Exercise 8 — GitHub

After downloading this repository, create a GitHub repository and push it:

```bash
git init
git add .
git commit -m 'Build TradeField agent POC'
git branch -M main
git remote add origin https://github.com/YOUR_USER/tradefield-agent.git
git push -u origin main
```

Never commit `.env` or an API key.


## Next agentic-AI upgrades

**Level 2:** persistent job state + SQLite

**Level 3:** RAG over trade-business SOPs and approved catalogue data

**Level 4:** voice intake using speech-to-text → agent → text-to-speech

**Level 5:** real calendar integration with approval

**Level 6:** real ServiceM8 adapter behind an approval gate

**Level 7:** event-driven architecture with queues/retries

**Level 8:** evaluation harness with 100+ scenarios and agent trajectory scoring
